# Teste o modelo criado:

## Bibliotecas Necessárias:

In [ ]:
import pandas as pd

import nltk
from nltk import tokenize

!pip install unidecode
import unidecode

import joblib

In [2]:
tfidf_loaded = joblib.load('./modelos/tfidf_vectorizer.pkl')                          # deserealização
regressao_logistica_loaded = joblib.load('./modelos/modelo_regressao_logistica.pkl')  # deserealização

## Criando função para processar dados novos:

In [3]:
palavras_irrelevantes = nltk.corpus.stopwords.words('portuguese')

token_pontuacao = tokenize.WordPunctTokenizer()

stemmer = nltk.RSLPStemmer()

def processar_avaliacao(avaliacao):
    # Passo 1: tokenização:
    tokens = token_pontuacao.tokenize(avaliacao)

    # Passo 2: Remover palavras irrelevantes:
    frase_processada = [palavra for palavra in tokens if palavra.lower() not in palavras_irrelevantes]

    # Passo 3: Remover pontuação
    frase_processada = [palavra for palavra in frase_processada if palavra.isalpha()]

    # Passo 4: Remover acentuação
    frase_processada = [unidecode.unidecode(palavra) for palavra in frase_processada]

    # Passo 5: Steeming, já vai normalizar o texto 
    frase_processada = [stemmer.stem(palavra) for palavra in frase_processada]

    return ' '.join(frase_processada)

## Classificando novas avaliações:

In [4]:
novas_avaliacoes = [
    "Até que o sabor é bom. Muito melhor que tomar whey. Vale a pena para um lanche, ceia, dia corrido",
    "Nossa sério, absurdo, compro o produto com recorrência, dessa vez senti um gosto estranho e não reparei assim que comprei, mas ele está vencido, veio uma leva vencida desde MARÇO, sendo que foi entregue pra mim em julho.",
    "Bom custo benefício",
    "Saboroso e nutritivo. Tem me salvado nos dias de correria do trabalho, quando não dá para fazer um lanche. Me ajuda bastante com os treinos.",
    "Fiquei decepcionada com a situação, a embalagem chegou totalmente encharcada (deixaram a mercadoria na portaria do condomínio cheio de formiga e com a embalagem derretendo). De 12 unidades, 06 estavam estouradas e o resto derretendo pois a embalagem do produto é de papelão. Totalmente impróprio para o consumo!",
    "Gente, o meu chegou todo certinho, eu tava com medo de vir faltando um ou estar quase fora do prazo de validade, mas veio tudo perfeito, dentro do prazo, bem embalado, tudo tranquilo, recomendo demais",
    "Gosto muito desse produto, ótima qualidade, e o preço estava muito bom comparado ao que vemos no mercado.",
    "Esse produto veio na quantidade que eu não pedi e ainda veio com a data de validade próxima. Quero o reembolso dessa compra",
    "TODOS VIERAM COM VENCIMENTO DE 3 MESES ATRÁS!!!"
]

In [5]:
novas_avaliacoes_processadas = [processar_avaliacao(avaliacao) for avaliacao in novas_avaliacoes]

In [6]:
novas_avaliacoes_processadas

['sab bom melhor tom whey val pen lanch cei dia corr',
 'seri absurd compr produt recorrenc dess vez sent gost estranh rep assim compr venc vei lev venc desd marc send entreg pra mim julh',
 'bom cust benefici',
 'sabor nutri salv dia corr trabalh da faz lanch ajud bast trein',
 'fiq decepcion situaca embal cheg total encharc deix mercad port condomini chei formig embal derret unidad estour rest derret poi embal produt papela total impropri consum',
 'gent cheg tod cert tav med vir falt quas praz validad vei tud perfeit dentr praz bem embal tud tranquil recom demal',
 'gost dess produt otim qual prec bom compar vem merc',
 'produt vei quant ped aind vei dat validad prox quer reembols dess compr',
 'tod vier venc mes atr']

In [7]:
novas_avaliacoes_tfidf = tfidf_loaded.transform(novas_avaliacoes_processadas)

predicoes = regressao_logistica_loaded.predict(novas_avaliacoes_tfidf)

df_previsoes = pd.DataFrame({
    'Avaliacao': novas_avaliacoes,
    'Sentimento_previsto': predicoes
})

In [8]:
df_previsoes

,Avaliacao,Sentimento_previsto
0,Até que o sabor é bom. Muito melhor que tomar ...,positivo
1,"Nossa sério, absurdo, compro o produto com rec...",negativo
2,Bom custo benefício,positivo
3,Saboroso e nutritivo. Tem me salvado nos dias ...,positivo
4,"Fiquei decepcionada com a situação, a embalage...",negativo
5,"Gente, o meu chegou todo certinho, eu tava com...",positivo
6,"Gosto muito desse produto, ótima qualidade, e ...",positivo
7,Esse produto veio na quantidade que eu não ped...,negativo
8,TODOS VIERAM COM VENCIMENTO DE 3 MESES ATRÁS!!!,negativo
